In [15]:

import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

import pandas as pd
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification


# 1. Load the SOTA casual-text medical NER model and tokenizer
model_name = "edwardsj/biobert-smm4h-ade"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
Name tf.RaggedTensorSpec has already been registered for class tensorflow.python.ops.ragged.ragged_tensor.RaggedTensorSpec.

In [ ]:

# Create an NLP pipeline optimized for CPU (aggregation_strategy resolves token fragments)
nlp_ner = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple"
)

# 2. Example Reddit-style text input
reddit_posts = [
    "Started taking Lexapro last week for my anxiety. The first few days were fine, but now I have this insane insomnia and a constant pounding headache.",
    "Anyone else get terrible stomach cramps and nausea about an hour after taking Metformin?"
]

# 3. Process the text and format the output
for idx, post in enumerate(reddit_posts, 1):
    print(f"\n--- Processing Reddit Post #{idx} ---")
    print(f"Text: \"{post}\"")
    
    # Run CPU inference
    ner_results = nlp_ner(post)
    
    if not ner_results:
        print("No Drugs or ADEs detected.")
        continue
        
    # Display the structured results
    for entity in ner_results:
        word = entity['word']
        label = entity['entity_group']
        confidence = entity['score']
        print(f" -> Found [{label}]: '{word}' (Confidence: {confidence:.2%})")
